# 超参数调优

RL 训练对超参数非常敏感。本节通过真实实验数据，展示三个关键超参数对训练的影响：entropy_coeff、kl_loss_coef 和学习率。

---

## 1. entropy_coeff

entropy_coeff 控制 Entropy Bonus 的强度，直接影响模型的探索能力。

### 实验数据

以下使用同一个 SFT 模型（Qwen3-1.7B-Wordle-SFT），仅改变 entropy_coeff：

| entropy_coeff | 观测到的 entropy | 结果 |
|---------------|------------------|------|
| 0 | 约 0.54 → 0.15（step 75）| 持续下降，策略崩塌风险较高 |
| 0.001 | 约 0.56 → 0.48（step 25）| 轻度保护，仍需继续观察 |
| 0.002 | 0.534 → 0.451（step 25）→ 0.223（step 155）| `0701_1404` 稳定完成，correct 达 70% |
| 0.005 | 0.531 → 0.998（step 25）→ **3.721**（step 38）| `0630_1646` 熵快速飙升并停止 |
| 0.01 | 约 0.56 → **4.51**（step 25）| entropy bonus 明显过强 |

### 分析

> ⚠️ **entropy_coeff 的合适值受 SFT 模型状态、随机种子和硬件浮点差异影响**。本 recipe 使用已完整验证的 `0.002` 作为默认值；新环境先跑约 30 步观察趋势，如果 entropy 持续加速上升则下调，如果快速逼近 0 则结合 KL、学习率和 reward 一起排查。


```text
entropy_coeff 的影响

entropy_coeff=0 (无 bonus)
  pg_loss 完全主导 -> entropy 持续下降 -> 最终崩塌

entropy_coeff=0.01 (过强)
  entropy_bonus 压过 pg_loss -> entropy 飙升 -> 模型过度探索
  25步内 entropy 从 0.56 飙到 4.51，指数增长
  reward 上升但 correct 不涨 (靠 partial 而非猜中)

entropy_coeff=0.002 (适中)
  三种力平衡 -> entropy 缓慢下降到 ~0.2 稳定 -> 训练稳定
```

---

## 2. kl_loss_coef

kl_loss_coef 控制 KL 正则化强度，限制当前策略与参考策略之间的距离。

| kl_loss_coef | 效果 | 风险 |
|-------------|------|------|
| 0.001 | 轻微约束 | 可能不足以阻止漂移 |
| 0.01 | 中等约束 | 配合 entropy_coeff 使用 |
| 0.1 | 强约束 | 模型无法学习新策略 |

### KL loss 与 entropy 的关系

KL loss 和 entropy bonus 是互补的稳定性机制：

```text
KL loss: 拉回参考策略 (限制漂移)
  kl_loss = KL(pi || pi_ref)
  偏离越远 -> kl_loss 越大 -> 梯度拉回

entropy_bonus: 推高探索 (防止崩塌)
  entropy_bonus = -entropy_coeff * H(pi)
  熵越低 -> 惩罚越大 -> 梯度推高

两者协同: KL 限制方向, entropy 限制分布形状
```

---

## 3. 学习率（Actor LR）

学习率决定每次更新的步长，直接影响训练的稳定性和收敛速度。

| ACTOR_LR | 效果 | 风险 |
|----------|------|------|
| 1e-7 | 更新太慢 | 训练效率低 |
| 1e-6 | 缓慢稳定 | 当前配置，推荐 |
| 1e-5 | 更新较快 | 可能导致梯度爆炸 |
| 1e-4 | 更新过快 | 训练不稳定 |

> RL 的学习率通常比 SFT 低 1-2 个数量级，因为 RL 更新基于 noisy 的优势估计，需要更保守的步长。

### 学习率调度（LR Scheduler）

除了学习率本身，**学习率调度策略**对训练稳定性同样关键。verl 支持两种调度：

| 调度类型 | 效果 | 适用场景 |
|----------|------|----------|
| constant（默认） | LR 恒定不变 | 短训练、调试阶段 |
| cosine | LR 余弦退火，后期逐步降低 | 正式训练，防止后期过更新 |

> **实测发现**：使用 constant 调度时，reward 在 step 135 附近见顶后回落（越峰退化）；切换为 cosine 调度（`min_lr_ratio=0.1`, `lr_warmup_steps_ratio=0.03`）后，reward 持续爬升到 step 155，correct 突破 70%。**建议正式训练使用 cosine 调度**。

---

## 4. 超参数组合建议

基于实验数据，推荐以下组合：

```text
推荐超参数组合

组合 A (保守稳定):
  entropy_coeff = 0.002
  kl_loss_coef  = 0.001
  ACTOR_LR      = 1e-6
  lr_scheduler  = cosine
  min_lr_ratio  = 0.1
  save_freq     = 25

组合 B (更强约束):
  entropy_coeff = 0.002
  kl_loss_coef  = 0.01
  ACTOR_LR      = 1e-6
  lr_scheduler  = cosine
  min_lr_ratio  = 0.1
  save_freq     = 25

调参顺序建议:
1. 先用默认参数跑 50 步，观察 entropy 趋势
2. 如果 entropy 下降过快 -> 加 entropy_coeff
3. 如果 entropy 失控飙升 -> 降 entropy_coeff
4. 如果策略漂移过大 -> 加 kl_loss_coef
```

---

## 课后练习

1. （判断题）entropy_coeff=0（默认值）时，模型没有任何探索保护机制，可能导致策略崩塌。

2. （判断题）entropy_coeff=0.01 比 entropy_coeff=0.005 更安全，因为更强的探索保证。

3. （判断题）RL 的学习率通常比 SFT 低 1-2 个数量级。

4. （单选题）entropy_coeff=0.01 导致 entropy 飙升到 4.51，说明什么？
    A. KL 正则太强
    B. Entropy bonus 过强，压过了策略梯度
    C. 学习率太低
    D. 训练正常

5. （单选题）调参时应该先观察什么？
    A. 直接设置最大 entropy_coeff
    B. 先用默认参数跑 50 步，观察 entropy 趋势
    C. 先调学习率
    D. 先调 KL coef

6. （多选题）以下哪些超参数影响 RL 训练稳定性？
    A. entropy_coeff
    B. kl_loss_coef
    C. ACTOR_LR
    D. ROLLOUT_N

> 完成上面的练习后，再运行下方单元格查看参考答案和解析。

In [ ]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
course_root = repo_root if (repo_root / 'tutorials/rl_training_pipeline').is_dir() else repo_root.parent / 'cann-learning-hub'
answer_path = course_root / 'tutorials/rl_training_pipeline/04_tuning_and_troubleshooting/answer/04.02_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))